# MIC full run on ROSE (one pair, one seed)

Settings: GPU T4 x1, Internet on, inputs `rose-mic` and `mit-b5`, secret `GITHUB_TOKEN`.
Set `CFG` in cell 1, then **Save Version > Save & Run All (Commit)** so it runs in the
background (up to 12 h). Results end up in the version's Output under `results/`.

In [ ]:
REPO = "Muhammad-Zubair-Yousafzai/weedrefine-uda"
CFG = "bean_2019_to_2021_s0_10k"   # one of mic/configs/rose/*_10k.py, without .py
import os
# Kaggle's default matplotlib backend is missing in the MIC env; MIC only saves figures.
os.environ["MPLBACKEND"] = "Agg"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import os, subprocess
os.chdir("/kaggle/working")
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None   # public repo
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

def git(*args):
    # never print the token, even in errors
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError((r.stderr or r.stdout).replace(token or "\0", "***"))

if not os.path.isdir("weedrefine-uda/.git"):
    git("clone", "-q", url, "weedrefine-uda")
    git("-C", "weedrefine-uda", "remote", "set-url", "origin", f"https://github.com/{REPO}.git")
else:
    git("-C", "weedrefine-uda", "fetch", "-q", url, "main")
    git("-C", "weedrefine-uda", "reset", "-q", "--hard", "FETCH_HEAD")
os.chdir("/kaggle/working/weedrefine-uda")
!git log -1 --format='code version: %h %s'
assert os.path.isfile(f"mic/configs/rose/{CFG}.py"), f"config {CFG} not in the repo"

In [ ]:
# About 10 minutes. Must end with 'Setup done' and then print 2.
!bash scripts/kaggle_mic_setup.sh
!grep -c "ROSE patch" third_party/MIC/seg/mmseg/models/utils/visualization.py
# results parser runs in Kaggle's own Python
!pip install -q --no-deps -e .

In [ ]:
# The long one: ~9.5 h for 10k iterations on a T4. Full log is in the work dir.
!cd third_party/MIC/seg && /root/miniforge/envs/mic/bin/python run_experiments.py --config configs/rose/{CFG}.py > /kaggle/working/train_{CFG}.txt 2>&1; tail -n 30 /kaggle/working/train_{CFG}.txt

In [ ]:
# IoU per evaluation, final result, results/{CFG}.json and log copies.
!python scripts/mic_results.py --name {CFG} --out /kaggle/working/results
!ls -lh /kaggle/working/results